# 02 — XML Cache Verification

Notebook 01 already downloads and caches the verified XML (one network pass, not two).
This notebook **audits the cache**: every file exists, is well-formed XML, and contains `<infoTable>` rows — so notebook 03 never parses a cover page or an HTML error page again.

> Rule from the design discussion: notebooks never call each other — they pass data through `data/`.

In [ ]:
# --- setup: make src/ importable from notebooks/ ---
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# On Google Colab, uncomment:
# !pip install -q pyarrow requests pandas matplotlib
# then upload/clone the repo so that src/ sits next to notebooks/

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
from pathlib import Path
from xml.etree import ElementTree as ET
from src import config
from src.utils import load_parquet
from src.sec import looks_like_info_table

filings = load_parquet(config.FILINGS_PARQUET)
filings[["quarter", "xml_file", "local_xml_path"]]

In [ ]:
report = []
for _, f in filings.iterrows():
    p = Path(f["local_xml_path"])
    txt = p.read_text(encoding="utf-8")
    ok_xml = True
    try:
        ET.fromstring(txt)
    except ET.ParseError:
        ok_xml = False
    report.append({
        "quarter": f["quarter"],
        "file": p.name,
        "size_kb": round(p.stat().st_size / 1024, 1),
        "well_formed": ok_xml,
        "has_infoTable": looks_like_info_table(txt),
        "looks_like_html": txt.lstrip()[:6].lower().startswith("<html"),  # the old 403 symptom
    })
audit = pd.DataFrame(report)
audit

In [ ]:
assert audit["well_formed"].all(), "Malformed XML in cache — re-run notebook 01"
assert audit["has_infoTable"].all(), "A cover page slipped through — re-run notebook 01"
assert not audit["looks_like_html"].any(), "HTML error page cached (bad User-Agent?)"
print("cache verified: ", len(audit), "quarters ready for parsing")